In [2]:
import pymupdf4llm

# Конвертируем весь PDF в один Markdown-строку
md_text = pymupdf4llm.to_markdown("Mois.pdf")

In [3]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "Header_1"),
    ("##", "Header_2"),
    ("###", "Header_3"),
]

splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
sections = splitter.split_text(md_text)

In [5]:
SYSTEM_PROMPT = """Ты — эксперт по извлечению знаний. Твоя задача — превратить текст в структурированные факты для базы знаний.

КРИТИЧЕСКИЕ ПРАВИЛА ФИЛЬТРАЦИИ:
1. ИГНОРИРУЙ служебную информацию: оглавления, списки литературы, ссылки на другие разделы ("см. главу X"), организационные указания ("ЛР №...", "автор курса").
2. ИГНОРИРУЙ разделы, которые не содержат определений, свойств или правил (например, только заголовок "Содержание").
3. ТЕРМИНЫ должны быть чистыми. Вместо "Понятие бинарного отношения" пиши "Бинарное отношение".
4. Если в тексте НЕТ полезных концептов, верни пустой список: {"section_title": "...", "concepts": [], "summary": "skip"}."""
import json
import time
from openai import OpenAI

client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

def process_to_tree_json(sections):
    knowledge_tree = {
        "subject": "Интеллектуальные системы",
        "sections": []
    }
    
    for i, section in enumerate(sections):
        headers = [v for k, v in section.metadata.items() if "Header" in k]
        current_title = headers[-1] if headers else "Введение"
        parent_path = " > ".join(headers[:-1]) if len(headers) > 1 else "Корень"

        # Добавляем в промт поле эвристик (нестрогих правил/советов)
        prompt = f"""Сформируй узел дерева знаний для: '{current_title}' (Контекст: {parent_path}).
ТЕКСТ:
{section.page_content}

ВЫХОД (JSON):
{{
  "section_title": "{current_title}",
  "concepts": [
    {{
      "term": "чистое название",
      "definition": "суть",
      "type": "definition/property/operation",
      "examples": [],
      "rules": ["строгие правила"],
      "heuristics": ["советы, нестрогие наблюдения или эвристики"]
    }}
  ],
  "summary": "Краткая суть раздела"
}}"""

        try:
            response = client.chat.completions.create(
                model="qwen2.5-coder-7b-instruct",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.1
            )
            
            raw_res = response.choices[0].message.content.strip()
            clean_res = raw_res[raw_res.find("{"):raw_res.rfind("}")+1]
            section_data = json.loads(clean_res)
            
            # ПРОВЕРКА: Если модель нашла полезные концепты, добавляем в дерево
            if section_data.get("concepts") and len(section_data["concepts"]) > 0:
                knowledge_tree["sections"].append(section_data)
                print(f"[{i+1}/{len(sections)}] + Добавлен раздел: {current_title}")
            else:
                print(f"[{i+1}/{len(sections)}] - Пропущен мусорный раздел: {current_title}")
            
        except Exception as e:
            print(f"Ошибка в секции {i+1}: {e}")
            
    return knowledge_tree

# Запуск
tree_data = process_to_tree_json(sections[10:100])

with open('knowledge_tree.json', 'w', encoding='utf-16') as f:
    json.dump(tree_data, f, ensure_ascii=False, indent=4)

[1/90] + Добавлен раздел: **1.1.3.2. Понятие бесконечного множества**
[2/90] + Добавлен раздел: **1.1.4. Понятие пустого множества**
[3/90] + Добавлен раздел: **1.1.5.1. Понятие подмножества**
[4/90] + Добавлен раздел: **1.1.5.2. Понятие надмножества**
[5/90] + Добавлен раздел: **1.1.6. Операции над множествами**
[6/90] + Добавлен раздел: **1.1.6.1. Понятие объединения множеств**
[7/90] + Добавлен раздел: **1.1.6.2. Понятие пересечения множеств**
[8/90] + Добавлен раздел: **1.1.6.3. Понятие разности двух множеств**
[9/90] + Добавлен раздел: **1.1.6.4. Понятие симметрической разности двух множеств**
[10/90] + Добавлен раздел: **1.1.6.5. Понятие булеана множества**
[11/90] + Добавлен раздел: **1.2. Понятие связки**
[12/90] + Добавлен раздел: **1.2.1. Понятие синглетона**
[13/90] + Добавлен раздел: **1.2.2. Понятие пары**
[14/90] + Добавлен раздел: **1.2.3. Понятие тройки**
[15/90] + Добавлен раздел: **1.2.4. Понятие небинарной связки**
[16/90] + Добавлен раздел: **1.2.5. Ориентированные 

In [9]:
import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

INPUT_TREE = 'knowledge_tree.json'
MODEL_PATH = './hugg'

with open(INPUT_TREE, 'r', encoding='utf-16') as f:
    tree_data = json.load(f)

model = SentenceTransformer(MODEL_PATH)

def build_vector_db_from_tree(tree):
    documents = []
    metadata_list = []
    
    print("--- Анализ дерева знаний ---")
    
    for section in tree.get("sections", []):
        for concept in section.get("concepts", []):
            term = concept.get("term", "")
            definition = concept.get("definition", "")
            
            # Собираем все доп. поля, преобразуя элементы в строки
            rules = ". ".join([str(r) for r in concept.get("rules", [])])
            examples = ". ".join([str(e) for e in concept.get("examples", [])])
            heuristics = ". ".join([str(h) for h in concept.get("heuristics", [])])
            
            # ЧИСТЫЙ ТЕКСТ ДЛЯ ЭМБЕДДИНГА (Поиск)
            embed_text = f"{term} — это {definition}"
            documents.append(embed_text)
            
            # ПОЛНЫЙ КОНТЕКСТ ДЛЯ NLI (Проверка)
            rich_context = f"Термин: {term}. Определение: {definition}."
            if rules: rich_context += f" Правила: {rules}."
            if examples: rich_context += f" Примеры: {examples}."
            if heuristics: rich_context += f" Эвристики: {heuristics}."
            
            metadata_list.append({
                "term": term,
                "rich_context": rich_context 
            })

    print(f"--- Векторизация {len(documents)} узлов дерева ---")
    embeddings = model.encode(documents, show_progress_bar=True)
    
    faiss.normalize_L2(embeddings)
    embeddings = np.array(embeddings).astype('float32')

    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension) 
    index.add(embeddings)

    faiss.write_index(index, "tree_kb.faiss")
    with open("tree_metadata.json", "w", encoding="utf-16") as f:
        json.dump(metadata_list, f, ensure_ascii=False, indent=4)
    
    print("--- Векторная база готова! ---")

if __name__ == "__main__":
    build_vector_db_from_tree(tree_data)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ./hugg
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- Анализ дерева знаний ---
--- Векторизация 265 узлов дерева ---


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

--- Векторная база готова! ---


In [10]:
import json
import faiss
import torch
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("--- Загрузка моделей ---")
search_model = SentenceTransformer('./hugg')

nli_model_path = './bert'
tokenizer = AutoTokenizer.from_pretrained(nli_model_path)
nli_model = AutoModelForSequenceClassification.from_pretrained(nli_model_path)

index = faiss.read_index("tree_kb.faiss")
with open("tree_metadata.json", "r", encoding="utf-16") as f:
    metadata = json.load(f)

def check_hallucination(claim):
    # --- ШАГ 1: Поиск в векторной базе FAISS ---
    # Векторизуем утверждение пользователя
    query_vec = search_model.encode([claim]).astype('float32')
    faiss.normalize_L2(query_vec)
    
    # Ищем топ-3 наиболее похожих фрагмента из базы знаний
    k_matches = 3
    distances, indices = index.search(query_vec, k=k_matches)
    
    # Устанавливаем порог сходства (Threshold)
    # Если сходство ниже 0.86, считаем, что информации в базе нет
    SIMILARITY_THRESHOLD = 0.86 
    
    print(f"\n[Проверяемое утверждение]: {claim}")
    
    if distances[0][0] < SIMILARITY_THRESHOLD:
        print(f"--- Результат проверки ---")
        print(f"СТАТУС: Нет данных (Максимальное сходство: {distances[0][0]:.2f})")
        print("В базе знаний не найдено релевантной информации для проверки этого факта.")
        return "Нет данных"

    # Загружаем маппинг лейблов из конфигурации NLI-модели
    id2label = nli_model.config.id2label

    best_entail_prob = -1
    best_result_dict = None
    best_match_term = ""

    print("[Логический анализ найденных совпадений]:")
    
    # --- ШАГ 2: Поочередная проверка каждого найденного совпадения ---
    for i in range(k_matches):
        sim = distances[0][i]
        
        # Анализируем только те результаты, которые близки к лучшему совпадению
        if sim >= (SIMILARITY_THRESHOLD - 0.05):
            idx = indices[0][i]
            match = metadata[idx]
            
            # Извлекаем термин и его определение
            term = match.get('term', '').lower()
            context = match.get('rich_context', '') 
            
            # Приводим текст к естественному виду для лучшего понимания моделью
            natural_text = context.replace("Термин: ", "").replace(". Определение: ", " — это ")
            
            # --- СЕМАНТИЧЕСКИЙ ФИЛЬТР (Subject Match) ---
            # Проверяем, упоминается ли найденный термин в проверяемом утверждении.
            # Это защищает от ложных подтверждений похожих, но разных понятий.
            claim_lower = claim.lower()
            subject_match = term in claim_lower or any(word in claim_lower for word in term.split() if len(word) > 3)
            
            # Подаем пару (Эталон, Утверждение) в NLI-модель
            inputs = tokenizer(natural_text, claim, truncation=True, max_length=512, return_tensors="pt")
            with torch.no_grad():
                outputs = nli_model(**inputs)
                probs = torch.softmax(outputs.logits, dim=1).tolist()[0]
            
            current_results = {}
            current_entail = 0
            
            for j, prob in enumerate(probs):
                label_name = id2label[j].lower()
                val = prob
                
                if 'entail' in label_name:
                    # ШТРАФ: Если термины не совпадают, снижаем вероятность подтверждения вдвое
                    if not subject_match:
                        val *= 0.5
                    current_entail = val
                    current_results['Подтверждено'] = val
                elif 'contradiction' in label_name:
                    current_results['Противоречие (Галлюцинация)'] = val
                else:
                    current_results['Нейтрально'] = val

            status_msg = "(Subject Match!)" if subject_match else "(Wrong Subject - Penalty applied)"
            print(f"  -> '{term}' | Подтверждение: {current_entail*100:.1f}% {status_msg}")
            
            # Выбираем тот фрагмент базы, который дает наиболее уверенное подтверждение
            if current_entail > best_entail_prob:
                best_entail_prob = current_entail
                best_result_dict = current_results
                best_match_term = term

    # --- ШАГ 3: Формирование итогового вердикта ---
    print(f"\n--- Итоговый результат (на основе '{best_match_term}') ---")
    
    if best_result_dict is None:
        return "Нет данных"

    # Сортируем результаты по вероятности для вывода
    sorted_res = sorted(best_result_dict.items(), key=lambda x: x[1], reverse=True)
    for label, prob in sorted_res:
        print(f"{label}: {prob*100:.2f}%")
    
    return max(best_result_dict, key=best_result_dict.get)

# Можешь запустить свои тесты здесь

# --- ТЕСТЫ ---
print("\n--- Тест 1: Правильное утверждение ---")
check_hallucination("Неориентированная связка связка, в которой элементы имеют одинаковые роли или не имеют их вовсе.")

print("\n--- Тест 2: Галлюцинация ---")
check_hallucination("Бинарное отношение — это множество пар, где первый элемент из одного множества, а второй — из другого.")

print("\n--- Тест 3: Правильное утверждение (Синонимы) ---")
# Проверяем определение ИС из раздела 1
check_hallucination("Интеллектуальная система — это программный комплекс, способный справляться с интеллектуальными задачами.")

print("\n--- Тест 4: Галлюцинация (Прямое противоречие) ---")
# Проверяем против факта: "Любой формальный язык задаётся при помощи языка математики"
check_hallucination("Математический язык не используется для описания формальных языков.")

print("\n--- Тест 5: Правильное утверждение (Атомарный факт) ---")
# Проверяем информацию о Лабораторной работе №4 из оглавления
check_hallucination("Лабораторная работа №4 посвящена разработке программ на языке асинхронного процедурного программирования SCP.")

print("\n--- Тест 6: Галлюцинация (Подмена понятий) ---")
# Проверяем против факта: "Мощность множества — это число элементов этого множества"
check_hallucination("Мощность множества — это название сущностей, из которых состоит данное множество.")

print("\n--- Тест 7: Правильное утверждение (Сложное определение) ---")
# Проверяем определение конечного множества из раздела 1.1.3.1
check_hallucination("Конечное множество — это такое множество, мощность которого равна некоторому неотрицательному целому числу.")

--- Загрузка моделей ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ./hugg
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: ./bert
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Тест 1: Правильное утверждение ---

[Проверяемое утверждение]: Неориентированная связка связка, в которой элементы имеют одинаковые роли или не имеют их вовсе.
[Логический анализ найденных совпадений]:
  -> 'неориентированная связка' | Подтверждение: 98.8% (Subject Match!)
  -> 'неориентированная связка' | Подтверждение: 4.0% (Subject Match!)
  -> 'ориентированная связка' | Подтверждение: 0.5% (Subject Match!)

--- Итоговый результат (на основе 'неориентированная связка') ---
Подтверждено: 98.78%
Нейтрально: 0.98%
Противоречие (Галлюцинация): 0.27%

--- Тест 2: Галлюцинация ---

[Проверяемое утверждение]: Бинарное отношение — это множество пар, где первый элемент из одного множества, а второй — из другого.
[Логический анализ найденных совпадений]:
  -> 'бинарное отношение' | Подтверждение: 4.6% (Subject Match!)
  -> 'бинарное отношение' | Подтверждение: 21.4% (Subject Match!)
  -> 'отношение' | Подтверждение: 3.0% (Subject Match!)

--- Итоговый результат (на основе 'бинарное отнош

'Подтверждено'